# Phase 9 — Optimization & Gradient Methods

## H&M Personalized Fashion Recommendations → Customer Purchase Prediction

### Objective

Phase 8 compared multiple model families and identified promising nonlinear models.

Phase 9 focuses on the **optimization side of machine learning**.

The goal is to understand and experimentally compare:

- Batch Gradient Descent
- Stochastic Gradient Descent
- Mini-batch Gradient Descent
- Learning-rate effects
- Regularization
- Logistic loss
- `SGDClassifier`
- Convergence behavior
- Learning curves
- Optimization vs generalization

This phase is important for a core-ML placement project because it demonstrates that the project is not just:

```text
data → sklearn.fit() → score
```

Instead, we explicitly investigate **how model parameters are optimized**.

---

## Core idea

For a parameterized model with loss function \(J(\theta)\):

\[
\theta_{t+1}
=
\theta_t
-
\eta \nabla J(\theta_t)
\]

where:

- \(\theta\) = model parameters
- \(\eta\) = learning rate
- \(\nabla J(\theta_t)\) = gradient of the loss

We will experimentally study how the choice of optimization strategy and learning rate affects convergence and generalization.

> The temporal train/validation split from Phase 5 is preserved throughout this phase.


# 9.1 Why Gradient Methods Matter

Many machine-learning models are trained by minimizing an objective function.

For binary classification, Logistic Regression commonly uses **log loss**:

\[
J(\theta)
=
-
\frac{1}{n}
\sum_{i=1}^{n}
[
y_i\log(p_i)
+
(1-y_i)\log(1-p_i)
]
\]

where:

\[
p_i = \sigma(x_i^T\theta)
\]

and

\[
\sigma(z)=\frac{1}{1+e^{-z}}
\]

The optimizer tries to find parameters that minimize this loss.

Different gradient strategies use different amounts of training data for each update.


# 9.2 Batch vs Stochastic vs Mini-Batch Gradient Descent

## Batch Gradient Descent

Uses the complete training set for one parameter update.

```text
Entire dataset
      ↓
compute gradient
      ↓
update parameters
```

**Advantages**
- Stable gradient estimate
- Smooth convergence

**Disadvantages**
- Expensive for very large datasets
- Each update requires processing the entire dataset

---

## Stochastic Gradient Descent

Uses one training example per update.

```text
one example
    ↓
gradient
    ↓
update
```

**Advantages**
- Very frequent updates
- Works well with huge datasets
- Can escape shallow local regions because of noisy updates

**Disadvantages**
- Noisy convergence
- Can oscillate around the optimum

---

## Mini-Batch Gradient Descent

Uses a small batch of examples.

```text
batch of examples
       ↓
gradient
       ↓
parameter update
```

This is the most common practical strategy for large-scale machine learning.


# 9.3 Imports and Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import time
import json
import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt

from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss
)

RANDOM_STATE = 42

PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = PROCESSED_DIR / "train_phase5.parquet"
VAL_PATH = PROCESSED_DIR / "validation_phase5.parquet"
TEST_PATH = PROCESSED_DIR / "test_phase5.parquet"

PREPROCESSOR_PATH = (
    MODELS_DIR / "preprocessor_standard_phase6.joblib"
)

print("Project root:", PROJECT_ROOT.resolve())


# 9.4 Load Phase 5 Data

In [ ]:
train_df = pd.read_parquet(TRAIN_PATH)
val_df = pd.read_parquet(VAL_PATH)
test_df = pd.read_parquet(TEST_PATH)

TARGET_COL = "target"
ID_COL = "customer_id"

X_train_raw = train_df.drop(
    columns=[TARGET_COL, ID_COL]
)
X_val_raw = val_df.drop(
    columns=[TARGET_COL, ID_COL]
)
X_test_raw = test_df.drop(
    columns=[TARGET_COL, ID_COL]
)

y_train = train_df[TARGET_COL].astype("int8")
y_val = val_df[TARGET_COL].astype("int8")
y_test = test_df[TARGET_COL].astype("int8")

print("Train:", X_train_raw.shape)
print("Validation:", X_val_raw.shape)
print("Test:", X_test_raw.shape)

print("\nPositive rate:")
print("Train:", y_train.mean())
print("Validation:", y_val.mean())
print("Test:", y_test.mean())


# 9.5 Load the Phase 6 Preprocessing Pipeline

We reuse the preprocessing pipeline fitted in Phase 6.

This ensures that:

- imputation is consistent,
- categorical encoding is consistent,
- feature ordering is consistent,
- validation/test information does not influence fitting of preprocessing parameters.


In [ ]:
preprocessor = joblib.load(
    PREPROCESSOR_PATH
)

X_train = preprocessor.transform(X_train_raw)
X_val = preprocessor.transform(X_val_raw)
X_test = preprocessor.transform(X_test_raw)

print("Train processed:", X_train.shape)
print("Validation processed:", X_val.shape)
print("Test processed:", X_test.shape)
print("Sparse matrix:", sp.issparse(X_train))


# 9.6 Why SGD is Appropriate for This Dataset

The H&M dataset can contain a very large number of customer records and one-hot encoded features.

A full dense matrix can be unnecessarily expensive.

`SGDClassifier` is particularly useful because it can train directly on sparse matrices.

Therefore, it is a practical way to study stochastic and mini-batch-style optimization on this dataset without forcing the entire matrix into dense memory.


# 9.7 Evaluation Helper

In [ ]:
def evaluate_classifier(
    model_name,
    split_name,
    model,
    X,
    y
):
    start = time.perf_counter()

    probabilities = model.predict_proba(X)[:, 1]

    prediction_time = time.perf_counter() - start

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    return {
        "model": model_name,
        "split": split_name,
        "accuracy": accuracy_score(
            y, predictions
        ),
        "precision": precision_score(
            y, predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y, predictions,
            zero_division=0
        ),
        "f1": f1_score(
            y, predictions,
            zero_division=0
        ),
        "roc_auc": roc_auc_score(
            y, probabilities
        ),
        "pr_auc": average_precision_score(
            y, probabilities
        ),
        "log_loss": log_loss(
            y, probabilities
        ),
        "prediction_time_seconds": prediction_time
    }


# 9.8 Experiment 1 — Logistic Regression Reference

We first train a standard Logistic Regression model.

This acts as the optimization reference.

The model is trained using a conventional solver rather than stochastic gradient descent.


In [ ]:
logistic_reference = LogisticRegression(
    penalty="l2",
    C=1.0,
    solver="liblinear",
    max_iter=1000,
    random_state=RANDOM_STATE
)

start = time.perf_counter()

logistic_reference.fit(
    X_train,
    y_train
)

logistic_fit_time = time.perf_counter() - start

logistic_reference_results = pd.DataFrame([
    evaluate_classifier(
        "LogisticRegression_Reference",
        "train",
        logistic_reference,
        X_train,
        y_train
    ),
    evaluate_classifier(
        "LogisticRegression_Reference",
        "validation",
        logistic_reference,
        X_val,
        y_val
    )
])

logistic_reference_results[
    "fit_time_seconds"
] = logistic_fit_time

display(logistic_reference_results)


# 9.9 Experiment 2 — SGDClassifier

`SGDClassifier(loss="log_loss")` gives us a logistic-regression-style classifier trained using stochastic gradient updates.

The important parameters are:

- `learning_rate`
- `eta0`
- `max_iter`
- `batching/update behavior`
- `alpha`

`alpha` controls L2 regularization when `penalty="l2"` is used.

This experiment lets us study the effect of optimization settings directly.


In [ ]:
sgd_default = SGDClassifier(
    loss="log_loss",
    penalty="l2",
    alpha=1e-4,
    learning_rate="optimal",
    max_iter=30,
    tol=1e-3,
    early_stopping=False,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

start = time.perf_counter()

sgd_default.fit(
    X_train,
    y_train
)

sgd_default_fit_time = time.perf_counter() - start

sgd_default_results = pd.DataFrame([
    evaluate_classifier(
        "SGD_LogLoss_Default",
        "train",
        sgd_default,
        X_train,
        y_train
    ),
    evaluate_classifier(
        "SGD_LogLoss_Default",
        "validation",
        sgd_default,
        X_val,
        y_val
    )
])

sgd_default_results[
    "fit_time_seconds"
] = sgd_default_fit_time

display(sgd_default_results)


# 9.10 Understanding the Learning Rate

The learning rate controls the size of each parameter update.

Conceptually:

\[
\theta_{t+1}
=
\theta_t
-
\eta \nabla J(\theta_t)
\]

### If learning rate is too small

```text
Very small steps
      ↓
slow convergence
      ↓
may stop before reaching a good solution
```

### If learning rate is too large

```text
Very large steps
      ↓
overshooting
      ↓
unstable / oscillating optimization
```

### Good learning rate

```text
stable updates
      ↓
reasonable convergence speed
      ↓
good final generalization
```

We will experimentally compare several learning-rate schedules.


# 9.11 Learning-Rate Experiment

We compare:

- constant learning rate
- inverse-scaling learning rate
- adaptive learning rate
- optimal learning rate

The experiment is deliberately controlled by keeping most other hyperparameters fixed.


In [ ]:
learning_rate_configs = [
    {
        "name": "constant_0.001",
        "learning_rate": "constant",
        "eta0": 0.001
    },
    {
        "name": "constant_0.01",
        "learning_rate": "constant",
        "eta0": 0.01
    },
    {
        "name": "constant_0.1",
        "learning_rate": "constant",
        "eta0": 0.1
    },
    {
        "name": "invscaling_0.01",
        "learning_rate": "invscaling",
        "eta0": 0.01
    },
    {
        "name": "adaptive_0.01",
        "learning_rate": "adaptive",
        "eta0": 0.01
    },
    {
        "name": "optimal",
        "learning_rate": "optimal",
        "eta0": 0.0
    }
]

learning_rate_results = []
learning_rate_models = {}

for config in learning_rate_configs:

    model = SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=1e-4,
        learning_rate=config["learning_rate"],
        eta0=config["eta0"],
        max_iter=30,
        tol=1e-3,
        early_stopping=False,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    start = time.perf_counter()

    model.fit(
        X_train,
        y_train
    )

    fit_time = time.perf_counter() - start

    learning_rate_models[
        config["name"]
    ] = model

    row = evaluate_classifier(
        config["name"],
        "validation",
        model,
        X_val,
        y_val
    )

    row["fit_time_seconds"] = fit_time
    row["learning_rate"] = config["learning_rate"]
    row["eta0"] = config["eta0"]

    learning_rate_results.append(row)

learning_rate_df = pd.DataFrame(
    learning_rate_results
)

display(
    learning_rate_df.sort_values(
        "pr_auc",
        ascending=False
    )
)


# 9.12 Plot Learning-Rate Performance

In [ ]:
plot_df = learning_rate_df.sort_values(
    "pr_auc",
    ascending=False
)

fig, ax = plt.subplots(
    figsize=(12, 6)
)

ax.bar(
    plot_df["model"],
    plot_df["pr_auc"]
)

ax.set_ylabel("Validation PR-AUC")
ax.set_xlabel("Learning-rate configuration")
ax.set_title(
    "Learning Rate vs Validation PR-AUC"
)

plt.xticks(
    rotation=35,
    ha="right"
)

plt.tight_layout()
plt.show()


# 9.13 Number of Epochs / Passes Through Data

For SGD-based training, `max_iter` approximately controls the maximum number of passes through the training data.

Too few iterations can cause underfitting:

```text
not enough optimization
       ↓
high training loss
       ↓
weak validation performance
```

Too many iterations can:

```text
increase training time
       ↓
provide diminishing returns
       ↓
potentially increase overfitting
```

We compare several iteration budgets.


In [ ]:
iteration_configs = [5, 10, 20, 40, 80]

iteration_results = []
iteration_models = {}

for n_iter in iteration_configs:

    model = SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=1e-4,
        learning_rate="optimal",
        max_iter=n_iter,
        tol=None,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    start = time.perf_counter()

    model.fit(
        X_train,
        y_train
    )

    fit_time = time.perf_counter() - start

    iteration_models[n_iter] = model

    train_row = evaluate_classifier(
        f"SGD_iter_{n_iter}",
        "train",
        model,
        X_train,
        y_train
    )

    val_row = evaluate_classifier(
        f"SGD_iter_{n_iter}",
        "validation",
        model,
        X_val,
        y_val
    )

    train_row["max_iter"] = n_iter
    val_row["max_iter"] = n_iter

    train_row["fit_time_seconds"] = fit_time
    val_row["fit_time_seconds"] = fit_time

    iteration_results.extend(
        [train_row, val_row]
    )

iteration_df = pd.DataFrame(
    iteration_results
)

display(iteration_df)


# 9.14 Plot Convergence with Number of Iterations

In [ ]:
iter_validation = iteration_df[
    iteration_df["split"] == "validation"
].sort_values("max_iter")

fig, ax = plt.subplots(
    figsize=(10, 6)
)

ax.plot(
    iter_validation["max_iter"],
    iter_validation["pr_auc"],
    marker="o",
    label="Validation PR-AUC"
)

ax.plot(
    iter_validation["max_iter"],
    iter_validation["f1"],
    marker="o",
    label="Validation F1"
)

ax.set_xlabel("Maximum iterations")
ax.set_ylabel("Score")
ax.set_title(
    "Optimization Budget vs Validation Performance"
)

ax.legend()

plt.tight_layout()
plt.show()


# 9.15 Regularization Experiment

Regularization controls model complexity.

For L2 regularization:

\[
J_{regularized}
=
J_{loss}
+
\frac{\lambda}{2}
\|\theta\|_2^2
\]

In `SGDClassifier`, the corresponding strength is controlled by `alpha`.

### Small alpha

Less regularization:

```text
more flexible model
      ↓
potentially lower training error
      ↓
higher overfitting risk
```

### Large alpha

More regularization:

```text
simpler parameter values
      ↓
lower variance
      ↓
possible underfitting
```

We will compare several values.


In [ ]:
alpha_values = [
    1e-6,
    1e-5,
    1e-4,
    1e-3,
    1e-2
]

regularization_results = []
regularization_models = {}

for alpha in alpha_values:

    model = SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=alpha,
        learning_rate="optimal",
        max_iter=30,
        tol=1e-3,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    start = time.perf_counter()

    model.fit(
        X_train,
        y_train
    )

    fit_time = time.perf_counter() - start

    regularization_models[
        alpha
    ] = model

    train_row = evaluate_classifier(
        f"SGD_alpha_{alpha}",
        "train",
        model,
        X_train,
        y_train
    )

    val_row = evaluate_classifier(
        f"SGD_alpha_{alpha}",
        "validation",
        model,
        X_val,
        y_val
    )

    train_row["alpha"] = alpha
    val_row["alpha"] = alpha

    train_row["fit_time_seconds"] = fit_time
    val_row["fit_time_seconds"] = fit_time

    regularization_results.extend(
        [train_row, val_row]
    )

regularization_df = pd.DataFrame(
    regularization_results
)

display(regularization_df)


# 9.16 Analyze Bias–Variance Through Regularization

In [ ]:
reg_train = regularization_df[
    regularization_df["split"] == "train"
].set_index("alpha")

reg_val = regularization_df[
    regularization_df["split"] == "validation"
].set_index("alpha")

reg_gap = pd.DataFrame({
    "alpha": alpha_values,
    "train_f1": [
        reg_train.loc[a, "f1"]
        for a in alpha_values
    ],
    "validation_f1": [
        reg_val.loc[a, "f1"]
        for a in alpha_values
    ],
    "f1_gap": [
        reg_train.loc[a, "f1"]
        -
        reg_val.loc[a, "f1"]
        for a in alpha_values
    ],
    "train_pr_auc": [
        reg_train.loc[a, "pr_auc"]
        for a in alpha_values
    ],
    "validation_pr_auc": [
        reg_val.loc[a, "pr_auc"]
        for a in alpha_values
    ],
    "pr_auc_gap": [
        reg_train.loc[a, "pr_auc"]
        -
        reg_val.loc[a, "pr_auc"]
        for a in alpha_values
    ]
})

display(reg_gap)


# 9.17 Plot Regularization vs Performance

In [ ]:
fig, ax = plt.subplots(
    figsize=(10, 6)
)

ax.semilogx(
    reg_gap["alpha"],
    reg_gap["train_pr_auc"],
    marker="o",
    label="Train PR-AUC"
)

ax.semilogx(
    reg_gap["alpha"],
    reg_gap["validation_pr_auc"],
    marker="o",
    label="Validation PR-AUC"
)

ax.set_xlabel("Regularization strength (alpha)")
ax.set_ylabel("PR-AUC")
ax.set_title(
    "Regularization Strength vs PR-AUC"
)

ax.legend()

plt.tight_layout()
plt.show()


# 9.18 Early Stopping Experiment

Early stopping is another form of regularization.

The idea:

```text
Training
  ↓
validation performance improves
  ↓
continue
  ↓
validation performance stops improving
  ↓
stop training
```

This can prevent unnecessary optimization after the model has already reached a useful solution.

We use `SGDClassifier`'s built-in validation-based early stopping.


In [ ]:
early_stopping_model = SGDClassifier(
    loss="log_loss",
    penalty="l2",
    alpha=1e-4,
    learning_rate="optimal",
    max_iter=100,
    tol=1e-3,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

start = time.perf_counter()

early_stopping_model.fit(
    X_train,
    y_train
)

early_stopping_fit_time = (
    time.perf_counter() - start
)

early_stopping_results = pd.DataFrame([
    evaluate_classifier(
        "SGD_EarlyStopping",
        "train",
        early_stopping_model,
        X_train,
        y_train
    ),
    evaluate_classifier(
        "SGD_EarlyStopping",
        "validation",
        early_stopping_model,
        X_val,
        y_val
    )
])

early_stopping_results[
    "fit_time_seconds"
] = early_stopping_fit_time

display(early_stopping_results)

print(
    "Actual iterations:",
    early_stopping_model.n_iter_
)


# 9.19 Class Imbalance and Gradient Optimization

The target distribution should be checked before interpreting SGD results.

When the positive class is relatively rare, an optimizer can obtain deceptively good accuracy by favoring the majority class.

For this reason we continue to emphasize:

- PR-AUC
- Recall
- Precision
- F1

rather than accuracy alone.

We also compare an SGD model using `class_weight="balanced"`.


In [ ]:
balanced_sgd = SGDClassifier(
    loss="log_loss",
    penalty="l2",
    alpha=1e-4,
    learning_rate="optimal",
    max_iter=30,
    tol=1e-3,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

start = time.perf_counter()

balanced_sgd.fit(
    X_train,
    y_train
)

balanced_sgd_fit_time = (
    time.perf_counter() - start
)

balanced_sgd_results = pd.DataFrame([
    evaluate_classifier(
        "SGD_Balanced",
        "train",
        balanced_sgd,
        X_train,
        y_train
    ),
    evaluate_classifier(
        "SGD_Balanced",
        "validation",
        balanced_sgd,
        X_val,
        y_val
    )
])

balanced_sgd_results[
    "fit_time_seconds"
] = balanced_sgd_fit_time

display(balanced_sgd_results)


# 9.20 Mini-Batch Gradient Descent from Scratch

`SGDClassifier` abstracts away the optimization loop.

To understand the algorithm itself, we implement a small **mini-batch Logistic Regression optimizer from scratch**.

The implementation is intentionally educational.

We will use a controlled numeric subset rather than the full high-dimensional one-hot matrix.

This allows us to directly observe:

```text
forward pass
     ↓
logistic prediction
     ↓
loss
     ↓
gradient
     ↓
parameter update
```

The full production model will continue to use sparse optimized libraries.


In [ ]:
# Select a compact numerical feature subset for educational
# from-scratch optimization.

numeric_candidates = [
    c for c in X_train_raw.columns
    if pd.api.types.is_numeric_dtype(
        X_train_raw[c]
    )
]

print(
    "Number of numeric candidate features:",
    len(numeric_candidates)
)

display(
    pd.DataFrame({
        "feature": numeric_candidates
    }).head(30)
)


In [ ]:
# Use the first available numeric features for the
# educational optimizer. The exact number is configurable.

SCRATCH_FEATURE_LIMIT = min(
    10,
    len(numeric_candidates)
)

scratch_features = numeric_candidates[
    :SCRATCH_FEATURE_LIMIT
]

X_scratch_train = X_train_raw[
    scratch_features
].copy()

X_scratch_val = X_val_raw[
    scratch_features
].copy()

# Numeric preprocessing for the scratch experiment.
scratch_medians = X_scratch_train.median()

X_scratch_train = (
    X_scratch_train
    .fillna(scratch_medians)
)

X_scratch_val = (
    X_scratch_val
    .fillna(scratch_medians)
)

# Standardize using TRAINING statistics only.
scratch_mean = X_scratch_train.mean()
scratch_std = (
    X_scratch_train.std()
    .replace(0, 1)
)

X_scratch_train = (
    (X_scratch_train - scratch_mean)
    / scratch_std
)

X_scratch_val = (
    (X_scratch_val - scratch_mean)
    / scratch_std
)

X_scratch_train = X_scratch_train.to_numpy(
    dtype=np.float64
)

X_scratch_val = X_scratch_val.to_numpy(
    dtype=np.float64
)

y_scratch_train = y_train.to_numpy(
    dtype=np.float64
)

y_scratch_val = y_val.to_numpy(
    dtype=np.float64
)

print(
    "Scratch train matrix:",
    X_scratch_train.shape
)


# 9.21 Mathematical Functions for Scratch Logistic Regression

For logistic regression:

\[
z = X\theta + b
\]

\[
p = \sigma(z)
=
\frac{1}{1+e^{-z}}
\]

The binary cross-entropy loss is:

\[
J =
-
\frac{1}{m}
\sum
[
y\log(p)
+
(1-y)\log(1-p)
]
\]

The gradients are:

\[
\frac{\partial J}{\partial \theta}
=
\frac{1}{m}X^T(p-y)
\]

\[
\frac{\partial J}{\partial b}
=
\frac{1}{m}
\sum(p-y)
\]

The parameters are updated using:

\[
\theta
\leftarrow
\theta
-
\eta
\frac{\partial J}{\partial\theta}
\]

\[
b
\leftarrow
b
-
\eta
\frac{\partial J}{\partial b}
\]


In [ ]:
def sigmoid(z):
    z = np.clip(z, -50, 50)
    return 1.0 / (1.0 + np.exp(-z))


def binary_cross_entropy(y_true, probability):
    probability = np.clip(
        probability,
        1e-12,
        1 - 1e-12
    )

    return -np.mean(
        y_true * np.log(probability)
        +
        (1 - y_true)
        * np.log(1 - probability)
    )


def predict_probability(
    X,
    weights,
    bias
):
    return sigmoid(
        X @ weights + bias
    )


# 9.22 Mini-Batch Gradient Descent Implementation

The optimizer below:

1. shuffles training observations,
2. creates mini-batches,
3. computes predictions,
4. computes the gradient,
5. updates weights,
6. records training loss,
7. records validation loss.

This lets us visualize convergence directly.


In [ ]:
def mini_batch_logistic_regression(
    X_train,
    y_train,
    X_val,
    y_val,
    learning_rate=0.01,
    batch_size=256,
    epochs=20,
    l2_strength=0.0,
    random_state=42
):
    rng = np.random.default_rng(
        random_state
    )

    n_samples, n_features = X_train.shape

    weights = np.zeros(
        n_features,
        dtype=np.float64
    )

    bias = 0.0

    history = []

    for epoch in range(epochs):

        indices = rng.permutation(
            n_samples
        )

        X_shuffled = X_train[indices]
        y_shuffled = y_train[indices]

        for start_idx in range(
            0,
            n_samples,
            batch_size
        ):

            end_idx = min(
                start_idx + batch_size,
                n_samples
            )

            X_batch = X_shuffled[
                start_idx:end_idx
            ]

            y_batch = y_shuffled[
                start_idx:end_idx
            ]

            probabilities = predict_probability(
                X_batch,
                weights,
                bias
            )

            errors = (
                probabilities - y_batch
            )

            grad_w = (
                X_batch.T @ errors
                / len(y_batch)
            )

            grad_b = np.mean(errors)

            # L2 regularization
            if l2_strength > 0:
                grad_w += (
                    l2_strength * weights
                )

            weights -= (
                learning_rate * grad_w
            )

            bias -= (
                learning_rate * grad_b
            )

        train_prob = predict_probability(
            X_train,
            weights,
            bias
        )

        val_prob = predict_probability(
            X_val,
            weights,
            bias
        )

        history.append({
            "epoch": epoch + 1,
            "train_loss": binary_cross_entropy(
                y_train,
                train_prob
            ),
            "validation_loss": binary_cross_entropy(
                y_val,
                val_prob
            )
        })

    return weights, bias, pd.DataFrame(history)


# 9.23 Run Mini-Batch Optimization

In [ ]:
scratch_start = time.perf_counter()

scratch_weights, scratch_bias, scratch_history = (
    mini_batch_logistic_regression(
        X_scratch_train,
        y_scratch_train,
        X_scratch_val,
        y_scratch_val,
        learning_rate=0.01,
        batch_size=256,
        epochs=20,
        l2_strength=1e-4,
        random_state=RANDOM_STATE
    )
)

scratch_fit_time = (
    time.perf_counter() - scratch_start
)

display(
    scratch_history.head()
)

print(
    "Scratch optimizer time:",
    scratch_fit_time,
    "seconds"
)


# 9.24 Plot Mini-Batch Convergence

This is one of the most important visualizations of Phase 9.

We directly observe how the objective changes across optimization epochs.


In [ ]:
fig, ax = plt.subplots(
    figsize=(10, 6)
)

ax.plot(
    scratch_history["epoch"],
    scratch_history["train_loss"],
    marker="o",
    label="Train loss"
)

ax.plot(
    scratch_history["epoch"],
    scratch_history["validation_loss"],
    marker="o",
    label="Validation loss"
)

ax.set_xlabel("Epoch")
ax.set_ylabel("Binary cross-entropy loss")
ax.set_title(
    "Mini-Batch Gradient Descent Convergence"
)

ax.legend()

plt.tight_layout()
plt.show()


# 9.25 Batch-Size Experiment

Batch size controls the amount of data used to estimate each gradient.

### Small batch

- noisier updates,
- more updates per epoch,
- potentially faster initial progress.

### Large batch

- smoother gradients,
- fewer updates per epoch,
- potentially higher memory and computational cost.

We compare several batch sizes using the scratch optimizer.


In [ ]:
batch_sizes = [
    32,
    128,
    256,
    1024
]

batch_histories = {}
batch_summary = []

for batch_size in batch_sizes:

    start = time.perf_counter()

    weights, bias, history = (
        mini_batch_logistic_regression(
            X_scratch_train,
            y_scratch_train,
            X_scratch_val,
            y_scratch_val,
            learning_rate=0.01,
            batch_size=batch_size,
            epochs=15,
            l2_strength=1e-4,
            random_state=RANDOM_STATE
        )
    )

    elapsed = time.perf_counter() - start

    batch_histories[
        batch_size
    ] = history

    batch_summary.append({
        "batch_size": batch_size,
        "final_train_loss": history[
            "train_loss"
        ].iloc[-1],
        "final_validation_loss": history[
            "validation_loss"
        ].iloc[-1],
        "fit_time_seconds": elapsed
    })

batch_summary_df = pd.DataFrame(
    batch_summary
)

display(batch_summary_df)


# 9.26 Plot Batch-Size Convergence

We compare validation loss trajectories for different batch sizes.


In [ ]:
fig, ax = plt.subplots(
    figsize=(11, 7)
)

for batch_size, history in batch_histories.items():

    ax.plot(
        history["epoch"],
        history["validation_loss"],
        marker="o",
        label=f"Batch {batch_size}"
    )

ax.set_xlabel("Epoch")
ax.set_ylabel("Validation loss")
ax.set_title(
    "Mini-Batch Size vs Validation-Loss Convergence"
)

ax.legend()

plt.tight_layout()
plt.show()


# 9.27 Learning-Rate × Batch-Size Experiment

Learning rate and batch size interact.

A learning rate that works for one batch size may behave differently for another.

We therefore perform a small controlled grid experiment.

This is intentionally small because exhaustive optimization belongs in a later hyperparameter-tuning phase.


In [ ]:
lr_batch_configs = [
    (0.001, 128),
    (0.005, 128),
    (0.01, 128),
    (0.001, 512),
    (0.005, 512),
    (0.01, 512)
]

lr_batch_results = []

for lr, batch_size in lr_batch_configs:

    start = time.perf_counter()

    weights, bias, history = (
        mini_batch_logistic_regression(
            X_scratch_train,
            y_scratch_train,
            X_scratch_val,
            y_scratch_val,
            learning_rate=lr,
            batch_size=batch_size,
            epochs=15,
            l2_strength=1e-4,
            random_state=RANDOM_STATE
        )
    )

    elapsed = time.perf_counter() - start

    lr_batch_results.append({
        "learning_rate": lr,
        "batch_size": batch_size,
        "final_train_loss": history[
            "train_loss"
        ].iloc[-1],
        "final_validation_loss": history[
            "validation_loss"
        ].iloc[-1],
        "fit_time_seconds": elapsed
    })

lr_batch_df = pd.DataFrame(
    lr_batch_results
)

display(
    lr_batch_df.sort_values(
        "final_validation_loss"
    )
)


# 9.28 Compare Optimization Strategies

We now assemble the main SGD experiments.

The table answers:

- Does SGD match the conventional Logistic Regression baseline?
- Which learning-rate schedule performs best?
- How much does regularization change performance?
- Does class balancing improve recall at the cost of precision?


In [ ]:
main_optimization_results = pd.concat(
    [
        logistic_reference_results,
        sgd_default_results,
        early_stopping_results,
        balanced_sgd_results
    ],
    ignore_index=True
)

display(
    main_optimization_results[
        [
            "model",
            "split",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc",
            "log_loss",
            "fit_time_seconds"
        ]
    ]
)


# 9.29 Compare Best SGD Configuration with Logistic Regression

We identify the strongest SGD configuration using validation PR-AUC.

The test set is deliberately excluded.

This is still an intermediate optimization experiment, not final model selection.


In [ ]:
best_lr_row = learning_rate_df.sort_values(
    "pr_auc",
    ascending=False
).iloc[0]

best_lr_name = best_lr_row["model"]

best_lr_model = learning_rate_models[
    best_lr_name
]

print(
    "Best learning-rate configuration:",
    best_lr_name
)

print(
    "Validation PR-AUC:",
    best_lr_row["pr_auc"]
)

print(
    "Validation F1:",
    best_lr_row["f1"]
)


# 9.30 Optimization vs Generalization

Optimization performance and generalization performance are not the same thing.

A model may:

```text
minimize training loss very effectively
             ↓
but
             ↓
perform poorly on future data
```

This is why we inspect:

- training loss,
- validation loss,
- training metrics,
- validation metrics.

The ideal region is where validation performance is strong without a large train-validation gap.


In [ ]:
reg_best = regularization_df[
    regularization_df["split"] == "validation"
].sort_values(
    "pr_auc",
    ascending=False
)

display(
    reg_best[
        [
            "alpha",
            "pr_auc",
            "roc_auc",
            "f1",
            "precision",
            "recall",
            "log_loss"
        ]
    ]
)


# 9.31 Extract SGD Coefficients

Because SGD with logistic loss is still a linear classifier, its coefficients can be inspected.

Large positive coefficients indicate features associated with increasing predicted log-odds of purchase.

Large negative coefficients indicate the opposite direction.

This gives us an interpretable bridge between:

```text
optimization
      ↓
learned parameters
      ↓
feature contribution
```


In [ ]:
sgd_feature_names = preprocessor.get_feature_names_out()

coefficient_df = pd.DataFrame({
    "feature": sgd_feature_names,
    "coefficient": best_lr_model.coef_[0]
})

coefficient_df["abs_coefficient"] = (
    coefficient_df["coefficient"].abs()
)

top_positive = (
    coefficient_df
    .sort_values(
        "coefficient",
        ascending=False
    )
    .head(20)
)

top_negative = (
    coefficient_df
    .sort_values(
        "coefficient",
        ascending=True
    )
    .head(20)
)

print("Top positive coefficients:")
display(top_positive)

print("Top negative coefficients:")
display(top_negative)


# 9.32 Save Optimization Results

Artifacts produced by this phase:

```text
results/
├── phase9_learning_rate_results.csv
├── phase9_iteration_results.csv
├── phase9_regularization_results.csv
├── phase9_batch_size_results.csv
├── phase9_learning_rate_batch_results.csv
└── phase9_optimization_summary.csv

models/
├── sgd_best_learning_rate_phase9.joblib
├── sgd_balanced_phase9.joblib
└── sgd_early_stopping_phase9.joblib
```


In [ ]:
learning_rate_df.to_csv(
    RESULTS_DIR / "phase9_learning_rate_results.csv",
    index=False
)

iteration_df.to_csv(
    RESULTS_DIR / "phase9_iteration_results.csv",
    index=False
)

regularization_df.to_csv(
    RESULTS_DIR / "phase9_regularization_results.csv",
    index=False
)

batch_summary_df.to_csv(
    RESULTS_DIR / "phase9_batch_size_results.csv",
    index=False
)

lr_batch_df.to_csv(
    RESULTS_DIR / "phase9_learning_rate_batch_results.csv",
    index=False
)

main_optimization_results.to_csv(
    RESULTS_DIR / "phase9_optimization_summary.csv",
    index=False
)

coefficient_df.to_csv(
    RESULTS_DIR / "phase9_sgd_coefficients.csv",
    index=False
)

joblib.dump(
    best_lr_model,
    MODELS_DIR / "sgd_best_learning_rate_phase9.joblib"
)

joblib.dump(
    balanced_sgd,
    MODELS_DIR / "sgd_balanced_phase9.joblib"
)

joblib.dump(
    early_stopping_model,
    MODELS_DIR / "sgd_early_stopping_phase9.joblib"
)

print("Phase 9 artifacts saved.")


# 9.33 Save Experiment Configuration

In [ ]:
phase9_config = {
    "random_state": RANDOM_STATE,
    "optimization_methods": [
        "Logistic Regression reference",
        "SGDClassifier with log loss",
        "Mini-batch Logistic Regression from scratch"
    ],
    "learning_rate_schedules": [
        "constant",
        "invscaling",
        "adaptive",
        "optimal"
    ],
    "learning_rates_tested": [
        0.001,
        0.01,
        0.1
    ],
    "regularization_alpha_values": alpha_values,
    "iteration_values": iteration_configs,
    "batch_sizes": batch_sizes,
    "scratch_feature_limit": SCRATCH_FEATURE_LIMIT,
    "test_set_used_for_selection": False
}

with open(
    RESULTS_DIR / "phase9_experiment_config.json",
    "w"
) as f:
    json.dump(
        phase9_config,
        f,
        indent=4
    )

print("Configuration saved.")


# 9.34 Final Interpretation Checklist

After running the notebook, answer these questions.

## Optimization

1. Does SGD converge faster than conventional Logistic Regression?
2. Which learning-rate schedule works best?
3. What happens when the learning rate is too large?
4. What happens when it is too small?
5. How does increasing `max_iter` affect performance?

## Regularization

6. Which `alpha` gives the best validation PR-AUC?
7. Does stronger regularization reduce the train-validation gap?
8. At what point does regularization cause underfitting?

## Mini-batch learning

9. How does batch size affect convergence?
10. Which batch size gives the best validation loss?
11. Does a smaller batch produce noisier optimization?

## Generalization

12. Does the model with the lowest training loss also have the best validation score?
13. Is there evidence of overfitting?
14. Does class balancing improve recall?
15. What happens to precision when recall increases?

## Placement-level takeaway

You should be able to explain:

> **Gradient descent is an optimization algorithm, not a model.**

For example:

```text
Logistic Regression
       +
Log Loss
       +
Gradient-based optimizer
       ↓
learned parameters
```

And:

```text
Batch GD
    vs
SGD
    vs
Mini-batch GD
```

differ primarily in how much training data is used to estimate each gradient update.


# 9.35 Important Leakage Check

This phase must not use the test set to tune:

- learning rate,
- regularization,
- number of iterations,
- batch size,
- class weighting,
- early stopping configuration.

All optimization decisions are made using training and validation data.

The test set remains untouched.

Run this check before moving to Phase 10.


In [ ]:
print("Test set exists:", len(X_test) > 0)
print("Test labels loaded:", len(y_test) > 0)

print("\nPhase 9 rule:")
print(
    "Test set was not used for optimization/model selection."
)


# Phase 9 Complete

We have now moved from:

```text
model comparison
```

to:

```text
understanding how a model learns
```

The project now demonstrates:

- Logistic loss
- Gradient computation
- Batch vs stochastic vs mini-batch learning
- Learning-rate experiments
- Regularization
- Early stopping
- Class imbalance handling
- Convergence curves
- Optimization/generalization analysis
- A small gradient-descent implementation from scratch

---

# Phase 10 Preview — Ensemble Learning

Next we will focus specifically on **ensemble methods**.

Topics:

- Bagging
- Random Forest
- Extra Trees
- Boosting
- Gradient Boosting
- XGBoost
- Voting
- Stacking
- Hard vs soft voting
- Ensemble diversity
- Correlated vs uncorrelated errors
- Comparing individual models against ensembles
- Ensemble performance vs computational cost

The key question will become:

> **Can combining different models produce a more accurate and robust predictor than any individual model?**
